## Outline

#### The purpose of this script is to fit nonlinear age effects to the mode activation data (using brms) and generate figures

In [2]:
# Import packages
import os
import mne
import re
import pandas as pd
import numpy as np
import glob
import datetime
from matplotlib import pyplot as plt
import matplotlib.gridspec as gridspec
import yaml
from nilearn import image, datasets
import seaborn as sns
from scipy.stats import zscore
from sklearn.preprocessing import StandardScaler

import arviz as az

# Imports for R functionality
from rpy2.robjects.packages import importr
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri

from rpy2.robjects import pandas2ri, default_converter
from rpy2.robjects.conversion import localconverter

# Explicitly define R functions
base = importr('base')
brms = importr('brms')
loo = importr('loo')
tidybayes = importr("tidybayes")
dplyr = importr("dplyr")
rstan = importr("rstan")
mgcv = importr("mgcv")

In [3]:
# Write out data?
writeOK=True

In [4]:
# Read config file and define paths
with open('0_config.yaml', 'r') as file:
    config = yaml.safe_load(file)
    
subjects_dir = config['dirs']['recon_dir']
proc_dir = config['dirs']['proc_dir']
atlas_dir = config['dirs']['atlas_dir']
model_dir = config['dirs']['model_dir']
results_dir = config['dirs']['results_dir']

# Update model dir for the appropriate training run (the one with the lowest variational free energy)
training_run = config['analysis_params']['selected_training_run']
model_dir = f'{model_dir}_run{training_run}'

# Define directory containing posthoc output
posthoc_dir = os.path.join(model_dir, 'posthoc')
os.makedirs(posthoc_dir, exist_ok=True)

# Define directory containing brms output
brm_dir = os.path.join(model_dir, 'brms_fits')
os.makedirs(brm_dir, exist_ok=True)

In [5]:
# Plotting params
colours = config['misc']['colours']
n_modes = 6

# Other params
orig_mode_order = config['posthoc']['orig_mode_order']
exclude = config['misc']['exclusion_col']
groups = config['misc']['groups']

orig_mode_order = config['posthoc']['orig_mode_order']
new_mode_order = config['posthoc']['new_mode_order']

now = datetime.datetime.now()
datetime_str = now.strftime("%Y-%m-%d_%H%M")

In [6]:
# Define a helper function to pull the most recent version of a file
# Note that this selects files based on the date & time that they were last
# modified, not the actual datetime printed in the filename 
def recent_fname(path):
    
    # Takes a partial file path
    files = glob.glob(path)
    most_recent_file = max(files, key=os.path.getmtime)
    
    return most_recent_file

In [6]:
# Files to load
subjects_fname = recent_fname(os.path.join(results_dir, 'subject_order_*.txt')) # this is correct for the new analysis
data_fname = recent_fname(os.path.join(posthoc_dir, '*_CNg_epoch_amplitudes_750ms.csv'))

# Load list of subjects that went into DyNeMo
subjects = np.loadtxt(subjects_fname, dtype=str).tolist()

# Read the CNg amplitudes for each mode
df_CNg = pd.read_csv(data_fname)

# compute mean mode amplitudes per subject, for plotting. 
df_CNg_avg = df_CNg.groupby(['SubjectID','Sex','Dx', 'UniqueID'], as_index=False).mean()

# Define functions

In [7]:
# Function to get peak (i.e., mode) posterior probability, using Arviz kde functionality
def get_az_mode(draws):

    # Ensure we are working with a clean numpy array
    x = np.asarray(draws)
    
    # az.kde returns (grid_points, density_values)
    grid, pdf = az.kde(x)
    
    # The mode is the grid point where density is highest
    return grid[np.argmax(pdf)]

In [8]:
# Function for estimating slope derivative at a given age (in z space)
def get_slope_at_age_z(fit, age_z, delta=0.1):

    # delta is in real age units, so convert to z space
    delta_z = delta/age_sd
   
    

    post_plus = brms.posterior_smooths(fit, smooth = "s(Age_z)", 
                                       newdata=ro.DataFrame({
                                           'Age_z': ro.FloatVector([age_z + delta_z])}), 
                                       re_formula = ro.r('NA'))
    
    post_minus = brms.posterior_smooths(fit, smooth = "s(Age_z)", 
                                       newdata=ro.DataFrame({
                                           'Age_z': ro.FloatVector([age_z - delta_z])}), 
                                       re_formula = ro.r('NA'))

    # Convert R matrices to NumPy arrays
    with localconverter(default_converter + pandas2ri.converter):
        post_plus_np = np.array(ro.conversion.rpy2py(post_plus))  
        post_minus_np = np.array(ro.conversion.rpy2py(post_minus))

    # Compute finite difference
    slope_post = (post_plus_np - post_minus_np) / (2 * delta_z)

    return slope_post

In [9]:
# Function to compute posterior distributions of slope derivatives across a set of pre-defined age bands, 
# and return summary stats (posterior mode and HDIs) for each band
def compute_band_posterior(fit, age_bands, mode_sd, age_sd):
    
    # Store posteriors for all bands and sexes
    band_posteriors = []
        
    # Loop through each age band
    for band_name, ages in age_bands.items():
        
        # Store slopes for all ages within this band
        slopes_list = []

        # Estimate slope for each age in this band
        for age in ages: 

            # Convert age to age_z
            age_z = (age - age_mean) / age_sd
            
            # Compute (a posterior distribution of) local slope derivative for each age
            slope_post = get_slope_at_age_z(fit, age_z) 

            # Back-transform the slope distribution to original scale
            slope_post_back = slope_post * (mode_sd / age_sd)

            # Append the posterior samples
            slopes_list.append(slope_post_back.flatten())

        # Take the average (across ages) draws across the band
        slopes_array = np.vstack(slopes_list)     # shape: (n_ages, n_draws)
        band_slopes = slopes_array.mean(axis=0)   # average across ages for each posterior draw

        # Create DataFrame for this band 
        df_band = pd.DataFrame({
            "Band": band_name,
            "Slope": band_slopes
        })
        
        # Append the dataframe for this band
        band_posteriors.append(df_band)

    # Combine all bands into one DataFrame
    posterior_df = pd.concat(band_posteriors, ignore_index=True)

    # Ensure correct ordering of bands
    posterior_df['Band'] = pd.Categorical(posterior_df['Band'], 
                                           categories=age_bands.keys(), 
                                           ordered=True)

    # Create a summary df with mode and HDI for each band
    bands_summary = (
    posterior_df.groupby(['Band'], observed=False).agg(
        mode_prob=('Slope', get_az_mode),  # note that this calls our custom function to get peak probability
        hdi_lower=('Slope', lambda x: az.hdi(x.to_numpy(), hdi_prob=0.95)[0]),
        hdi_upper=('Slope', lambda x: az.hdi(x.to_numpy(), hdi_prob=0.95)[1])
    ).reset_index()
    )

    return posterior_df, bands_summary


# Set up for brms

In [ ]:
# Z-transform Age and mode amplitude (we'll only model the z-scored data)
df_CNg['Age_z'] = zscore(df_CNg['Age'])
df_CNg['mode1_z'] = zscore(df_CNg['mode1'])
df_CNg['mode2_z'] = zscore(df_CNg['mode2'])
df_CNg['mode3_z'] = zscore(df_CNg['mode3'])
df_CNg['mode4_z'] = zscore(df_CNg['mode4'])
df_CNg['mode5_z'] = zscore(df_CNg['mode5'])
df_CNg['mode6_z'] = zscore(df_CNg['mode6'])

# Get original mean and SD for age 
age_mean = df_CNg['Age'].mean()
age_sd = df_CNg['Age'].std()

# Assign to R environment for brms
with localconverter(default_converter + pandas2ri.converter):
    r_df_CNg = ro.conversion.py2rpy(df_CNg)
ro.globalenv["df"] = r_df_CNg

In [ ]:
# Fitting params 
n_chains = 4
n_cores = 4
n_iter = 20000  
n_warmup = 5000  
control = ro.ListVector({"adapt_delta": 0.99})

# Define desired probability for HDIs
prob=0.95

# Define priors
priors = ro.r(
    'c('
        'brms::prior("normal(0, 1)", class = "Intercept"),'
        'brms::prior("normal(0, 1)", class = "sd"),'
        'brms::prior("normal(0, 0.25)", class = "sds"),'
        'brms::prior("normal(0, 1)", class = "b", coef = "sAge_z_1"),'  # linear component of the smooth age term
        'brms::prior("normal(0, 1)", class = "sigma")'
    ')'
)
#         

# Define age groups for which we will later estimate HDIs on smooth terms
age_bands = {'5-9': range(5, 9),
             '10-14': range(10, 14),
             '15-19': range(15, 19),
             '20-24': range(20, 24),
             '25-29': range(25, 29),
             '30-34': range(30, 34),
             '35-40': range(35, 40)
             }

# Estimate parameters for each mode
This will take some time

In [ ]:
# Fit a brms model for each mode
fits = []

for m in range(1,7):

    print(m)

    # Print out mode #
    print(f'Running model for {orig_mode_order[m-1]} Mode')
    
    # Fit the model
    fit = brms.brm(
            ro.Formula(f"mode{m}_z ~ s(Age_z) + (1 | SubjectID)"),
            data=r_df_CNg,
            prior=priors, 
            sample_prior="yes",
            chains=n_chains, iter=n_iter, warmup=n_warmup, cores=n_cores,
            control = control, backend = "cmdstan",
            refresh=5000
        )

    fits.append(fit)

    # Save the model and model summary to disk
    if writeOK:
        fit_fname = os.path.join(brm_dir, f'{datetime_str}_brms_mode{m}_750ms_amplitude_by_smooth_Age_Zspace')
        base.saveRDS(fit, file=f'{fit_fname}.rds')

        # Save the model summary to txt
        summary = str(brms.print_brmsfit(fit))
        with open(f'{fit_fname}_summary.txt', 'w') as f:
            f.write(summary)




In [ ]:
# Check prior summary for a given model, if desired
#ro.r['prior_summary'](fits[-1])

# Print summary for one model
# print(ro.r.summary(fits[5]))
#ro.r.plot(fits[5])



In [7]:
# Optional: Read the pre-computed models from disk
fits = []
for i in range(6):
    print(f"Diagnostics for {orig_mode_order[i]}")
    fit_fname = os.path.join(brm_dir,
                             recent_fname(os.path.join(brm_dir, f"*_brms_mode{i+1}_750ms_amplitude_by_smooth_Age_Zspace.rds")))
    fit = base.readRDS(fit_fname)

    # Check for any problems
    rstan.check_hmc_diagnostics(fit.rx2("fit"))
    
    fits.append(fit)  

Diagnostics for Frontotemporal

Divergences:


R callback write-console: 0 of 60000 iterations ended with a divergence.
  



Tree depth:


R callback write-console: 3 of 60000 iterations saturated the maximum tree depth of 10 (0.005%).
Try increasing 'max_treedepth' to avoid saturation.
  



Energy:


...

R callback write-console: 0 of 60000 iterations ended with a divergence.
  



Tree depth:


R callback write-console: 0 of 60000 iterations saturated the maximum tree depth of 10.
  



Energy:


R callback write-console: E-BFMI indicated no pathological behavior.
  


In [12]:
print(fits[0].rx2['prior'])

           prior     class      coef     group resp dpar nlpar lb ub tag
          (flat)         b                                              
    normal(0, 1)         b  sAge_z_1                                    
    normal(0, 1) Intercept                                              
    normal(0, 1)        sd                                      0       
    normal(0, 1)        sd           SubjectID                  0       
    normal(0, 1)        sd Intercept SubjectID                  0       
 normal(0, 0.25)       sds                                      0       
 normal(0, 0.25)       sds  s(Age_z)                            0       
    normal(0, 1)     sigma                                      0       
       source
      default
         user
         user
         user
 (vectorized)
 (vectorized)
         user
 (vectorized)
         user



In [ ]:
# Loop through fits, getting smooths for age, and back-transforming to original space
all_smooths = []

for f, fit in enumerate(fits):


    # Get mode mean and SD for back-transformation
    mode_mean = df_CNg[f'mode{f+1}'].mean()
    mode_sd = df_CNg[f'mode{f+1}'].std()

    # Use tidybayes in R to get HDIs
    ro.globalenv['current_fit'] = fit
    
    # Extract Age_z from the model data
    age_z = fit.rx2("data").rx2("Age_z")
    
    age_grid = ro.DataFrame({
        "Age_z": base.seq(base.min(age_z), base.max(age_z), **{"length.out": 100})
        })

    ro.globalenv["age_grid"] = age_grid

    # Get posterior draws
    draws =  ro.r('add_epred_draws(current_fit, newdata = age_grid, re_formula = NA)')
    ro.globalenv["draws"] = draws
    
    # Compute HDIs along the smooth term
    hdi_results = ro.r('mode_hdi(draws, .epred, .width = 0.95)')
    ro.globalenv["hdi_results"] = hdi_results

    # Summarize as a dataframe containing mode, upper HDI limit, lower HDI limit (for every step in the age grid)
    hdi_summary = ro.r('''
        data.frame(
            Age_z = hdi_results$Age_z,
            estimate__ = hdi_results$.epred,
            lower__ = hdi_results$.lower,
            upper__ = hdi_results$.upper
        )
        ''')
    

    # Convert to pandas
    with localconverter(default_converter + pandas2ri.converter):
        smooths_df = ro.conversion.rpy2py(hdi_summary)

    # Rename columns to indicate Z-space
    smooths_df = smooths_df.rename(columns={
        'estimate__': 'estimate__z',
        'lower__': 'lower__z',
        'upper__': 'upper__z'
    })  

    # Back-transform to original scale
    smooths_df['Age'] = (smooths_df['Age_z'] * age_sd) + age_mean  
    smooths_df['estimate__'] = (smooths_df['estimate__z'] * mode_sd) + mode_mean 
    smooths_df['lower__'] = (smooths_df['lower__z'] * mode_sd) + mode_mean
    smooths_df['upper__'] = (smooths_df['upper__z'] * mode_sd) + mode_mean 

    all_smooths.append(smooths_df)

In [ ]:
# Get band posteriors and HDIs for each mode
posterior_band_dfs = []
band_summaries = []
for mode in range(len(fits)):

    print(f'Mode {mode}')

    # Get mode SD for back-transformation
    mode_mean = df_CNg[f'mode{f+1}'].mean()
    mode_sd = df_CNg[f'mode{f+1}'].std()

    posterior_df, band_summary = compute_band_posterior(fits[mode], age_bands, mode_sd=mode_sd, age_sd=age_sd)

    posterior_band_dfs.append(posterior_df)
    band_summaries.append(band_summary)


# Plot results

In [ ]:
mode_order = [orig_mode_order.index(name) for name in new_mode_order]

# Reorder colours if needed
colours_new_order = [colours[i] for i in mode_order]

In [ ]:
# Create a figure with smooth terms in the left column and age-band summaries on the right
fig, axes = plt.subplots(
    nrows=len(fits), 
    ncols=2, 
    figsize=(10, 12),
    gridspec_kw={'width_ratios': [2, 1]}
)

# Modes in the new order
for plot_idx, mode in enumerate(mode_order):

    # Define axes
    ax_left = axes[plot_idx, 0]
    ax_right = axes[plot_idx, 1]

    # Call the model and summary stats for this mode
    fit = fits[mode]
    smooth_df = all_smooths[mode]
    band_summary = band_summaries[mode]
    mode_col = f'mode{mode+1}'

    # Plot smooth curve
    ax_left.plot(smooth_df['Age'], smooth_df['estimate__'], color=colours_new_order[plot_idx], linewidth=1.5)
    ax_left.fill_between(smooth_df['Age'], smooth_df['lower__'], smooth_df['upper__'],
                         color=colours_new_order[plot_idx], alpha=0.5)

    # Scatter raw points
    ax_left.scatter(df_CNg_avg['Age'], df_CNg_avg[mode_col], color=colours_new_order[plot_idx], s=10, alpha=0.8)


    # Optional: add a regplot for sanity checking
    # sns.regplot(
    #     data=df_coh_avg, 
    #     x='Age', 
    #     y=mode_col, 
    #     scatter=False, 
    #     scatter_kws={'s': 10, 'alpha': 0.5, 'color': 'blue'},
    #     line_kws={'color': 'k', 'alpha': 0.3, 'linestyle': '--'},
    #     ax=ax_left,
    #     order=1  # linear fit
    # )
    
    #ax_left.set_title(new_mode_names[plot_idx])
    ax_left.set_ylabel('Mode Activation')
    if plot_idx == len(fits)-1:
        ax_left.set_xlabel('Age (years)')
    else:
        ax_left.set_xlabel('')
        ax_left.set_xticklabels([])
    
    # A few outliers are really distorting  the y axis. Let's set the upper y axis limit to 
    # the 95th percentile of Y values
    ymin, ymax = np.percentile(df_CNg_avg[mode_col], [5, 95])
    ax_left.set_ylim(ymin, ymax)

    # Right axis plotting HDIs
    y_offsets = np.arange(len(age_bands))
    for row_idx, (_, row) in enumerate(band_summary.iterrows()):
        ax_right.plot(row['mode_prob'], y_offsets[row_idx], marker='d', color=colours_new_order[plot_idx])
        ax_right.hlines(y=y_offsets[row_idx], xmin=row['hdi_lower'], xmax=row['hdi_upper'],
                        color=colours_new_order[plot_idx], linewidth=2)

    ax_right.set_ylabel('Age Band')
    ax_right.set_yticks(y_offsets)
    ax_right.set_yticklabels(age_bands.keys())
    ax_right.set_xlim([-0.0025, 0.0025])
    ax_right.axvline(x=0, color='grey', linestyle='--', linewidth=1)
    if plot_idx == len(fits)-1:
        ax_right.set_xlabel('Slope Derivative')
    else:
        ax_right.set_xlabel('')
        ax_right.set_xticklabels([])


plt.tight_layout()
plt.subplots_adjust(hspace=0.5)
plt.savefig(os.path.join(model_dir, 'figures', f'mode_activation_age_smooths_and_band_HDI_summary.png'), dpi=300)
plt.show()


# Supplementary NHST analysis
Perform conventional spline regression with mgcv. For supplementary materials

In [ ]:
# Esnure df_CNg is accessible to the R environment
with localconverter(default_converter + pandas2ri.converter):
    r_df_CNg = ro.conversion.py2rpy(df_CNg)
ro.globalenv["df"] = r_df_CNg

# Make SubjectID a factor in R
ro.r('df$SubjectID <- as.factor(df$SubjectID)')

# Reassign df back to Python variable so gam() sees the updated subjectID factor
r_df_CNg = ro.globalenv['df']

# Loop through modes
for m in [1,2,3,4,5,6]:
    
    print(f'{orig_mode_order[m-1]} Mode')

    # fit the model and print the model summary
    fit = mgcv.gam(ro.Formula(f'mode{m}_z ~ s(Age_z) + s(SubjectID, bs="re")'),
             data = r_df_CNg)
    
    print(ro.r.summary(fit))